# 🧠 CS6886W — Assignment 1 : VGG6 on CIFAR-10

| | |
|---|---|
| **Course** | CS6886W – Deep Learning |
| **Assignment** | A1 – CNN Architecture Exploration |
| **Model** | VGG6 (custom, activation-injectable) |
| **Dataset** | CIFAR-10 (50 k train / 10 k test) |
| **Tracking** | Weights & Biases (W&B) |

> **Before running:** `Runtime → Change runtime type → T4 GPU`


In [12]:
import torch, platform
print('Python :', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌  No GPU found — switch runtime!')
assert torch.cuda.is_available(), 'GPU not available. Go to Runtime → Change runtime type → GPU'
print('\n✅ GPU is ready!')


Python : 3.13.16
PyTorch: 2.11.0+cu130
CUDA   : 13.0
GPU    : Tesla T4

✅ GPU is ready!


In [13]:
# Install required packages
!pip install -q wandb torch torchvision tqdm

# Clone the repo
import os
REPO_URL = 'https://github.com/cs25m506/vgg6-cifar10-assignment.git'  # ← update
REPO_DIR = '/content/vgg6-cifar10-assignment'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print('Repo already cloned — pulling latest...')
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
print('\n✅ Working directory:', os.getcwd())


Repo already cloned — pulling latest...
Already up to date.
/content

✅ Working directory: /content/vgg6-cifar10-assignment


In [14]:
import os

BASE_DIR    = '/content/vgg6-cifar10-assignment'
DATA_DIR    = f'{BASE_DIR}/data'
CKPT_DIR    = f'{BASE_DIR}/checkpoints'
RESULTS_DIR = f'{BASE_DIR}/results'

for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'✅  Data        → {DATA_DIR}')
print(f'✅  Checkpoints → {CKPT_DIR}')
print(f'✅  Results     → {RESULTS_DIR}')


✅  Data        → /content/vgg6-cifar10-assignment/data
✅  Checkpoints → /content/vgg6-cifar10-assignment/checkpoints
✅  Results     → /content/vgg6-cifar10-assignment/results


W&B is configured in *anonymous mode*


In [15]:
import wandb, os

# ── Anonymous mode (default — no key needed) ─────────────────────────
wandb.login(anonymous='allow')

WANDB_PROJECT = 'CS6886W-A1-VGG6-CIFAR10'
WANDB_ENTITY  = None   # keep None for anonymous; set to 'your_username' if logged in

print(f'✅  W&B project : {WANDB_PROJECT}')
print(f'    Entity      : {WANDB_ENTITY or "anonymous"}')


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


✅  W&B project : CS6886W-A1-VGG6-CIFAR10
    Entity      : anonymous


Alternative — Login with API Key (optional)


In [ ]:
# ── Run this cell ONLY if you want to use a named W&B account ────────
# Option A: Colab Secrets (recommended — key never printed)
# from google.colab import userdata
# os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

# Option B: Paste key directly (less secure)
# os.environ['WANDB_API_KEY'] = 'paste_your_key_here'

# wandb.login()
# WANDB_ENTITY = 'your_wandb_username'   # ← update
print('Skipped — using anonymous mode.')


In [16]:
import sys
sys.path.insert(0, '/content/vgg6-cifar10-assignment')

import torch, torch.nn as nn
import wandb
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from src.model             import VGG6, ACTIVATIONS
from src.dataset           import get_dataloaders
from src.engine            import train_one_epoch, evaluate
from src.optimizer_factory import build_optimizer, SUPPORTED_OPTIMIZERS
from src.utils             import set_seed, get_device, save_checkpoint
from configs               import BASE_CONFIG

SEED   = 42
set_seed(SEED)
DEVICE = get_device()
print(f'\n✅  Device: {DEVICE}  |  Seed: {SEED}')


[utils] Using device: cuda

✅  Device: cuda  |  Seed: 42


Augmentations applied during training (Q1-a):
- `RandomHorizontalFlip(p=0.5)`
- `RandomCrop(32, padding=4)`
- `Normalize(mean=(0.4914,0.4822,0.4465), std=(0.2470,0.2435,0.2616))`


In [17]:
train_loader, test_loader = get_dataloaders(
    data_dir   = DATA_DIR,
    batch_size = BASE_CONFIG['batch_size'],
    num_workers= 2,
)

imgs, labels = next(iter(train_loader))
print(f'Train batches : {len(train_loader)}  | batch shape: {imgs.shape}')
print(f'Test  batches : {len(test_loader)}  | batch shape: {imgs.shape}')

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

# Show sample grid
import torchvision
import torchvision.utils as vutils
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
STD  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)
imgs_show = imgs[:16].cpu() * STD + MEAN
grid = vutils.make_grid(imgs_show, nrow=8, padding=2, normalize=False)
fig, ax = plt.subplots(figsize=(12,3))
ax.imshow(grid.permute(1,2,0).clamp(0,1))
ax.set_title('Sample CIFAR-10 training images (after augmentation)', fontsize=12)
ax.axis('off')
fig.savefig(f'{RESULTS_DIR}/sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅  Sample grid saved →', f'{RESULTS_DIR}/sample_grid.png')


Train batches : 391  | batch shape: torch.Size([128, 3, 32, 32])
Test  batches : 79  | batch shape: torch.Size([128, 3, 32, 32])
✅  Sample grid saved → /content/vgg6-cifar10-assignment/results/sample_grid.png


In [18]:
def run_experiment(cfg: dict, notes: str = '') -> dict:
    '''
    Run one full training experiment, log to W&B, save best checkpoint.
    Returns a result dict with best_val_acc, run_name, run_url.
    '''
    set_seed(cfg.get('seed', SEED))

    # ── Dataloaders ──────────────────────────────────────────────────
    tr_loader, te_loader = get_dataloaders(
        data_dir   = DATA_DIR,
        batch_size = cfg['batch_size'],
        num_workers= 2,
    )

    # ── Model ────────────────────────────────────────────────────────
    model     = VGG6(num_classes=10, activation=cfg['activation']).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model.parameters(), cfg['optimizer'],
                                lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['num_epochs']
    )

    # ── W&B init ─────────────────────────────────────────────────────
    run_name = (f"act={cfg['activation']}_opt={cfg['optimizer']}"
               f"_bs={cfg['batch_size']}_lr={cfg['lr']}_ep={cfg['num_epochs']}")
    run = wandb.init(
        project  = WANDB_PROJECT,
        entity   = WANDB_ENTITY,
        name     = run_name,
        config   = cfg,
        notes    = notes,
        reinit   = True,
        anonymous= 'allow',
    )
    print(f'\n🚀  Run: {run_name}')
    print(f'📊  W&B: {run.url}')

    # ── Training loop ────────────────────────────────────────────────
    best_val_acc   = 0.0
    best_ckpt_path = os.path.join(CKPT_DIR, f'{run_name}_best.pth')
    history = {'epoch':[], 'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

    for epoch in tqdm(range(1, cfg['num_epochs']+1), desc=run_name):
        tr_loss, tr_acc   = train_one_epoch(model, tr_loader, criterion, optimizer, DEVICE, epoch, run)
        val_loss, val_acc = evaluate(model, te_loader, criterion, DEVICE, epoch, 'val', run)
        scheduler.step()

        history['epoch'].append(epoch)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint({'epoch':epoch,'model':model.state_dict(),
                             'optimizer':optimizer.state_dict(),
                             'val_acc':val_acc,'config':cfg}, best_ckpt_path)

    run.summary['best_val_acc'] = best_val_acc
    run.finish()
    print(f'\n✅  Best val acc: {best_val_acc:.2f}%  |  ckpt → {best_ckpt_path}')
    return {'run_name':run_name,'run_url':run.url,
            'best_val_acc':best_val_acc,'history':history,'config':cfg}


In [19]:
from google.colab import files as colab_files

def plot_and_save(histories: list, label_key: str, title: str,
                  filename: str, metric: str = 'val_acc',
                  ylabel: str = 'Validation Accuracy (%)'):
    '''
    Plot a metric for multiple runs, save PNG to RESULTS_DIR,
    log to W&B as an image, and offer a Colab download.
    '''
    fig, ax = plt.subplots(figsize=(10, 5))
    for r in histories:
        label = r['config'][label_key]
        ax.plot(r['history']['epoch'], r['history'][metric], marker='o',
                markersize=3, linewidth=1.8, label=str(label))
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()

    save_path = f'{RESULTS_DIR}/{filename}'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾  Saved → {save_path}')

    # Log image to W&B (inside an active run, or standalone)
    try:
        run_tmp = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                             name=f'plots_{filename}', job_type='plot',
                             reinit=True, anonymous='allow')
        run_tmp.log({title: wandb.Image(save_path)})
        run_tmp.finish()
    except Exception as e:
        print(f'W&B image log skipped: {e}')

    # Download in Colab
    colab_files.download(save_path)
    return save_path


def download_all_results():
    '''Zip and download all PNGs from RESULTS_DIR.'''
    zip_path = '/content/vgg6_results.zip'
    !zip -j $zip_path $RESULTS_DIR/*.png
    colab_files.download(zip_path)
    print(f'📦  Downloaded all result images as vgg6_results.zip')


Train VGG6 with the **best single configuration** determined from Q2 experiments.

| Param | Value |
|---|---|
| Activation | ReLU |
| Optimizer | Adam |
| LR | 1e-3 |
| Batch Size | 128 |
| Epochs | 30 |
| Seed | 42 |


In [20]:
q1_cfg = {**BASE_CONFIG,
          'data_dir':   DATA_DIR,
          'checkpoint_dir': CKPT_DIR,
          'activation': 'relu',
          'optimizer':  'adam',
          'lr':         1e-3,
          'batch_size': 128,
          'num_epochs': 30,
          'seed':       42}

q1_result = run_experiment(q1_cfg, notes='Q1 Baseline')
print(f'\n📌  Q1 Best Val Acc: {q1_result["best_val_acc"]:.2f}%')
print(f'📊  Dashboard      : {q1_result["run_url"]}')


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



🚀  Run: act=relu_opt=adam_bs=128_lr=0.001_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/fkm70ld6


act=relu_opt=adam_bs=128_lr=0.001_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇██
train/acc,▁▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████████
train/loss,█▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▃▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██████████
val/loss,█▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
best_val_acc,88.15
epoch,30
train/acc,89.76
train/loss,0.30717
val/acc,88.09
val/loss,0.36108



✅  Best val acc: 88.15%  |  ckpt → /content/vgg6-cifar10-assignment/checkpoints/act=relu_opt=adam_bs=128_lr=0.001_ep=30_best.pth

📌  Q1 Best Val Acc: 88.15%
📊  Dashboard      : https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/fkm70ld6


In [21]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
h = q1_result['history']

axes[0].plot(h['epoch'], h['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(h['epoch'], h['val_loss'],   label='Val Loss',   color='tomato')
axes[0].set_title('Q1 — Loss Curves', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(h['epoch'], h['train_acc'], label='Train Acc', color='steelblue')
axes[1].plot(h['epoch'], h['val_acc'],   label='Val Acc',   color='tomato')
axes[1].set_title('Q1 — Accuracy Curves', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle(f'Q1 Baseline (Best Val Acc: {q1_result["best_val_acc"]:.2f}%)', fontsize=13)
fig.tight_layout()
path = f'{RESULTS_DIR}/q1_baseline_curves.png'
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
from google.colab import files as colab_files
colab_files.download(path)
print('✅  Q1 plot downloaded!')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅  Q1 plot downloaded!


Compare **5 activation functions** keeping all other hyperparameters fixed:

| Activation | Description |
|---|---|
| `relu` | Rectified Linear Unit (baseline) |
| `sigmoid` | Logistic sigmoid |
| `tanh` | Hyperbolic tangent |
| `silu` | Sigmoid-weighted Linear Unit (Swish) |
| `gelu` | Gaussian Error Linear Unit |


In [22]:
ACTIVATIONS_LIST = ['relu', 'sigmoid', 'tanh', 'silu', 'gelu']
q2a_results = []

for act in ACTIVATIONS_LIST:
    cfg = {**BASE_CONFIG,
           'data_dir':       DATA_DIR,
           'checkpoint_dir': CKPT_DIR,
           'activation':     act,
           'optimizer':      'adam',
           'lr':             1e-3,
           'batch_size':     128,
           'num_epochs':     30,
           'seed':           42}
    result = run_experiment(cfg, notes=f'Q2a activation={act}')
    q2a_results.append(result)

print('\n📊  Q2a Summary:')
for r in q2a_results:
    print(f"  {r['config']['activation']:10s}  →  Best Val Acc: {r['best_val_acc']:.2f}%")



🚀  Run: act=relu_opt=adam_bs=128_lr=0.001_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/0n5ulmjw


act=relu_opt=adam_bs=128_lr=0.001_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/acc,▁▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████████
train/loss,█▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▃▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██████████
val/loss,█▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
best_val_acc,88.15
epoch,30
train/acc,89.76
train/loss,0.30717
val/acc,88.09
val/loss,0.36108



✅  Best val acc: 88.15%  |  ckpt → /content/vgg6-cifar10-assignment/checkpoints/act=relu_opt=adam_bs=128_lr=0.001_ep=30_best.pth



🚀  Run: act=sigmoid_opt=adam_bs=128_lr=0.001_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/9vlfwg0v


act=sigmoid_opt=adam_bs=128_lr=0.001_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

... (10 outputs left out)


🚀  Run: act=gelu_opt=adam_bs=128_lr=0.001_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/x64xr5vo


act=gelu_opt=adam_bs=128_lr=0.001_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
train/acc,▁▃▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████████
train/loss,█▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇█▇██████████
val/loss,█▆▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,89.44
epoch,30
train/acc,92.126
train/loss,0.2288
val/acc,89.44
val/loss,0.32405



✅  Best val acc: 89.44%  |  ckpt → /content/vgg6-cifar10-assignment/checkpoints/act=gelu_opt=adam_bs=128_lr=0.001_ep=30_best.pth

📊  Q2a Summary:
  relu        →  Best Val Acc: 88.15%
  sigmoid     →  Best Val Acc: 44.62%
  tanh        →  Best Val Acc: 82.23%
  silu        →  Best Val Acc: 89.14%
  gelu        →  Best Val Acc: 89.44%


In [23]:
plot_and_save(q2a_results, label_key='activation',
              title='Q2a — Validation Accuracy: Activation Comparison',
              filename='q2a_activation_val_acc.png',
              metric='val_acc', ylabel='Validation Accuracy (%)')

plot_and_save(q2a_results, label_key='activation',
              title='Q2a — Training Loss: Activation Comparison',
              filename='q2a_activation_train_loss.png',
              metric='train_loss', ylabel='Training Loss')

# Bar chart — best val acc per activation
acts   = [r['config']['activation'] for r in q2a_results]
accs   = [r['best_val_acc']         for r in q2a_results]
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(acts, accs, color=colors, edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fmt='%.2f%%', padding=3, fontsize=10)
ax.set_ylim(min(accs)-5, max(accs)+5)
ax.set_xlabel('Activation Function', fontsize=12)
ax.set_ylabel('Best Val Accuracy (%)', fontsize=12)
ax.set_title('Q2a — Best Val Accuracy per Activation', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
path = f'{RESULTS_DIR}/q2a_activation_bar.png'
fig.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
colab_files.download(path)


💾  Saved → /content/vgg6-cifar10-assignment/results/q2a_activation_val_acc.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾  Saved → /content/vgg6-cifar10-assignment/results/q2a_activation_train_loss.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Compare **4 optimizers** keeping activation=ReLU and all other hyperparameters fixed:

| Optimizer | Notes |
|---|---|
| `sgd` | Vanilla SGD + momentum=0.9 |
| `nesterov_sgd` | SGD + Nesterov momentum |
| `adam` | Adaptive moment estimation (baseline) |
| `adagrad` | Adaptive gradient |


In [24]:
OPTIMIZERS_LIST = ['sgd', 'nesterov_sgd', 'adam', 'adagrad']
q2b_results = []

for opt in OPTIMIZERS_LIST:
    lr = 0.01 if 'sgd' in opt else 1e-3   # SGD needs higher LR
    cfg = {**BASE_CONFIG,
           'data_dir':       DATA_DIR,
           'checkpoint_dir': CKPT_DIR,
           'activation':     'relu',
           'optimizer':      opt,
           'lr':             lr,
           'batch_size':     128,
           'num_epochs':     30,
           'seed':           42}
    result = run_experiment(cfg, notes=f'Q2b optimizer={opt}')
    q2b_results.append(result)

print('\n📊  Q2b Summary:')
for r in q2b_results:
    print(f"  {r['config']['optimizer']:15s}  →  Best Val Acc: {r['best_val_acc']:.2f}%")



🚀  Run: act=relu_opt=sgd_bs=128_lr=0.01_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/t6senl5a


act=relu_opt=sgd_bs=128_lr=0.01_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
train/acc,▁▃▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████████
train/loss,█▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/acc,▁▃▅▅▅▅▅▆▅▆▇▇▇▇▇▇▇▇▇▇▇█████████
val/loss,█▆▄▄▄▄▄▃▄▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
best_val_acc,87.63
epoch,30
train/acc,87.444
train/loss,0.36889
val/acc,87.63
val/loss,0.37642



✅  Best val acc: 87.63%  |  ckpt → /content/vgg6-cifar10-assignment/checkpoints/act=relu_opt=sgd_bs=128_lr=0.01_ep=30_best.pth



🚀  Run: act=relu_opt=nesterov_sgd_bs=128_lr=0.01_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/ds7carau


act=relu_opt=nesterov_sgd_bs=128_lr=0.01_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

... (14 outputs left out)


🚀  Run: act=relu_opt=nadam_bs=128_lr=0.001_ep=30
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/buj5uou2


act=relu_opt=nadam_bs=128_lr=0.001_ep=30:   0%|          | 0/30 [00:00<?, ?it/s]

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/acc,▁▃▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████████
train/loss,█▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val/acc,▁▄▄▄▄▆▅▅▅▆▇▆▆▇▇▇▇▇▇█▇█████████
val/loss,█▆▆▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁
best_val_acc,87.99
epoch,30
train/acc,90.286
train/loss,0.29684
val/acc,87.99
val/loss,0.36236



✅  Best val acc: 87.99%  |  ckpt → /content/vgg6-cifar10-assignment/checkpoints/act=relu_opt=nadam_bs=128_lr=0.001_ep=30_best.pth

📊  Q2b Summary:
  sgd              →  Best Val Acc: 87.63%
  nesterov_sgd     →  Best Val Acc: 87.57%
  adam             →  Best Val Acc: 88.15%
  adagrad          →  Best Val Acc: 73.58%
  rmsprop          →  Best Val Acc: 76.24%
  nadam            →  Best Val Acc: 87.99%


In [25]:
plot_and_save(q2b_results, label_key='optimizer',
              title='Q2b — Validation Accuracy: Optimizer Comparison',
              filename='q2b_optimizer_val_acc.png',
              metric='val_acc', ylabel='Validation Accuracy (%)')

plot_and_save(q2b_results, label_key='optimizer',
              title='Q2b — Training Loss: Optimizer Comparison',
              filename='q2b_optimizer_train_loss.png',
              metric='train_loss', ylabel='Training Loss')

# Bar chart
opts   = [r['config']['optimizer'] for r in q2b_results]
accs   = [r['best_val_acc']        for r in q2b_results]
fig, ax = plt.subplots(figsize=(9,4))
bars = ax.bar(opts, accs, color='#4C72B0', edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fmt='%.2f%%', padding=3, fontsize=10)
ax.set_ylim(min(accs)-5, max(accs)+5)
ax.set_xlabel('Optimizer', fontsize=12)
ax.set_ylabel('Best Val Accuracy (%)', fontsize=12)
ax.set_title('Q2b — Best Val Accuracy per Optimizer', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=15)
fig.tight_layout()
path = f'{RESULTS_DIR}/q2b_optimizer_bar.png'
fig.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
colab_files.download(path)


💾  Saved → /content/vgg6-cifar10-assignment/results/q2b_optimizer_val_acc.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾  Saved → /content/vgg6-cifar10-assignment/results/q2b_optimizer_train_loss.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Grid search over batch size, learning rate, and number of epochs:

| Param | Values |
|---|---|
| Batch Size | 64, 128, 256 |
| Learning Rate | 1e-4, 1e-3, 5e-3 |
| Epochs | 10, 20 |


In [28]:
import itertools

BATCH_SIZES = [64, 128]
LR_VALUES   = [1e-4, 1e-3]
EPOCH_LIST  = [10, 10]

q2c_results = []

for bs, lr, ep in itertools.product(BATCH_SIZES, LR_VALUES, EPOCH_LIST):
    cfg = {**BASE_CONFIG,
           'data_dir':       DATA_DIR,
           'checkpoint_dir': CKPT_DIR,
           'activation':     'relu',
           'optimizer':      'adam',
           'lr':             lr,
           'batch_size':     bs,
           'num_epochs':     ep,
           'seed':           42}
    result = run_experiment(cfg, notes=f'Q2c bs={bs} lr={lr} ep={ep}')
    q2c_results.append(result)

print('\n📊  Q2c Summary (top 5):')
sorted_q2c = sorted(q2c_results, key=lambda x: x['best_val_acc'], reverse=True)
for r in sorted_q2c[:5]:
    c = r['config']
    print(f"  bs={c['batch_size']:3d}  lr={c['lr']:.0e}  ep={c['num_epochs']:2d}  "
          f"→  {r['best_val_acc']:.2f}%")


🚀  Run: act=relu_opt=adam_bs=64_lr=0.0001_ep=10
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/lm83xd95


act=relu_opt=adam_bs=64_lr=0.0001_ep=10:   0%|          | 0/10 [00:00<?, ?it/s]

epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
train/acc,▁▄▅▆▇▇▇███
train/loss,█▅▄▃▂▂▂▁▁▁
val/acc,▁▄▆▆▆▇████
val/loss,█▅▃▃▃▂▁▁▁▁
best_val_acc,77.4
epoch,10
train/acc,73.642
train/loss,0.7612
val/acc,77.4
val/loss,0.64426



✅  Best val acc: 77.40%  |  ckpt → /content/vgg6-cifar10-assignment/checkpoints/act=relu_opt=adam_bs=64_lr=0.0001_ep=10_best.pth



🚀  Run: act=relu_opt=adam_bs=64_lr=0.0001_ep=10
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/p0gpte8j


act=relu_opt=adam_bs=64_lr=0.0001_ep=10:   0%|          | 0/10 [00:00<?, ?it/s]

... (10 outputs left out)


🚀  Run: act=relu_opt=adam_bs=64_lr=0.005_ep=10
📊  W&B: https://wandb.ai/balaji-kulkarni92/CS6886W-A1-VGG6-CIFAR10/runs/8vic60wf


act=relu_opt=adam_bs=64_lr=0.005_ep=10:   0%|          | 0/10 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a8c7e738c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1654, in _shutdown_workers
    self._worker_result_queue.put((None, None))
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 92, in put
    self._start_thread()
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 190, in _start_thread
    self._thread.start()
  File "/usr/lib/python3.13/threading.py", line 982, in start
    self._started.wait()  # Will set ident and native_id
  File "/usr/lib/python3.13/threading.py", line 660, in wait
    signaled = self._cond.wait(timeout)
  File "/usr/lib/python3.13/threading.py", line 359, in wait
    waiter.acquire()
KeyboardInterrupt: 


KeyboardInterrupt: 

In [32]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Build list of dicts for ep=10
rows = [{'bs': r['config']['batch_size'],
         'lr': r['config']['lr'],
         'acc': r['best_val_acc']}
        for r in q2c_results if r['config']['num_epochs'] == 10]

# Drop duplicates by keeping the latest run for each (bs, lr) combination
df_rows = pd.DataFrame(rows)
if not df_rows.empty:
    df_rows = df_rows.drop_duplicates(subset=['bs', 'lr'], keep='last')

df_pivot = df_rows.pivot(index='lr', columns='bs', values='acc')

fig, ax = plt.subplots(figsize=(7,4))
sns.heatmap(df_pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Val Acc (%)'})
ax.set_title('Q2c — Best Val Acc (%) | Epochs=10', fontsize=13, fontweight='bold')
ax.set_xlabel('Batch Size', fontsize=11)
ax.set_ylabel('Learning Rate', fontsize=11)
fig.tight_layout()
path = f'{RESULTS_DIR}/q2c_heatmap.png'
fig.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
colab_files.download(path)
print('✅  Q2c heatmap downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅  Q2c heatmap downloaded!


In [33]:
plot_and_save(q2a_results, label_key='activation',
              title='Q3 — Val Accuracy vs Epoch: Activation Functions',
              filename='q3_plot1_activations.png',
              metric='val_acc', ylabel='Validation Accuracy (%)')


💾  Saved → /content/vgg6-cifar10-assignment/results/q3_plot1_activations.png


epoch,▁▁▃▃▆▆█
train/acc,▁▃▅█
train/loss,█▅▃▁
val/acc,▁▆█
val/loss,█▁▁
epoch,4
train/acc,38.076
train/loss,1.61745
val/acc,32.95
val/loss,1.76466


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/vgg6-cifar10-assignment/results/q3_plot1_activations.png'

In [34]:
plot_and_save(q2b_results, label_key='optimizer',
              title='Q3 — Val Accuracy vs Epoch: Optimizers',
              filename='q3_plot2_optimizers.png',
              metric='val_acc', ylabel='Validation Accuracy (%)')


💾  Saved → /content/vgg6-cifar10-assignment/results/q3_plot2_optimizers.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/vgg6-cifar10-assignment/results/q3_plot2_optimizers.png'

In [35]:
plot_and_save(q2a_results, label_key='activation',
              title='Q3 — Training Loss vs Epoch: Activation Functions',
              filename='q3_plot3_act_loss.png',
              metric='train_loss', ylabel='Training Loss')


💾  Saved → /content/vgg6-cifar10-assignment/results/q3_plot3_act_loss.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/vgg6-cifar10-assignment/results/q3_plot3_act_loss.png'

In [36]:
plot_and_save(q2b_results, label_key='optimizer',
              title='Q3 — Training Loss vs Epoch: Optimizers',
              filename='q3_plot4_opt_loss.png',
              metric='train_loss', ylabel='Training Loss')


💾  Saved → /content/vgg6-cifar10-assignment/results/q3_plot4_opt_loss.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/vgg6-cifar10-assignment/results/q3_plot4_opt_loss.png'

In [37]:
all_results = q2a_results + q2b_results + q2c_results
rows = []
for r in all_results:
    rows.append({
        'Run'         : r['run_name'],
        'Activation'  : r['config']['activation'],
        'Optimizer'   : r['config']['optimizer'],
        'Batch Size'  : r['config']['batch_size'],
        'LR'          : r['config']['lr'],
        'Epochs'      : r['config']['num_epochs'],
        'Best Val Acc': f"{r['best_val_acc']:.2f}%",
        'W&B URL'     : r['run_url'],
    })
df_summary = pd.DataFrame(rows)
df_summary = df_summary.sort_values('Best Val Acc', ascending=False).reset_index(drop=True)
display(df_summary)

# Save CSV
csv_path = f'{RESULTS_DIR}/results_summary.csv'
df_summary.to_csv(csv_path, index=False)
colab_files.download(csv_path)
print('✅  Summary CSV downloaded!')


,Run,Activation,Optimizer,Batch Size,LR,Epochs,Best Val Acc,W&B URL
0,act=gelu_opt=adam_bs=128_lr=0.001_ep=30,gelu,adam,128,0.0010,30,89.44%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
1,act=silu_opt=adam_bs=128_lr=0.001_ep=30,silu,adam,128,0.0010,30,89.14%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
2,act=relu_opt=adam_bs=128_lr=0.001_ep=30,relu,adam,128,0.0010,30,88.15%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
3,act=relu_opt=adam_bs=128_lr=0.001_ep=30,relu,adam,128,0.0010,30,88.15%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
4,act=relu_opt=nadam_bs=128_lr=0.001_ep=30,relu,nadam,128,0.0010,30,87.99%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
5,act=relu_opt=sgd_bs=128_lr=0.01_ep=30,relu,sgd,128,0.0100,30,87.63%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
6,act=relu_opt=nesterov_sgd_bs=128_lr=0.01_ep=30,relu,nesterov_sgd,128,0.0100,30,87.57%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
7,act=tanh_opt=adam_bs=128_lr=0.001_ep=30,tanh,adam,128,0.0010,30,82.23%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
8,act=relu_opt=adam_bs=64_lr=0.001_ep=10,relu,adam,64,0.0010,10,81.48%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...
9,act=relu_opt=adam_bs=64_lr=0.001_ep=10,relu,adam,64,0.0010,10,81.48%,https://wandb.ai/balaji-kulkarni92/CS6886W-A1-...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅  Summary CSV downloaded!


In [38]:
fig, ax = plt.subplots(figsize=(8,5))
for bs in BATCH_SIZES:
    subset = [r for r in q2c_results if r['config']['batch_size'] == bs]
    lrs  = [r['config']['lr']        for r in subset]
    accs = [r['best_val_acc']        for r in subset]
    ax.scatter(lrs, accs, s=80, label=f'BS={bs}', zorder=3)

ax.set_xscale('log')
ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('Best Val Accuracy (%)', fontsize=12)
ax.set_title('Q3 — LR vs Best Val Accuracy (Q2c)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.tight_layout()
path = f'{RESULTS_DIR}/q3_scatter_lr_acc.png'
fig.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
colab_files.download(path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
# Run this cell in a fresh session to pull history from WandB
# Requires either an API key or anonymous run URLs
# ─────────────────────────────────────────────────────────
# import wandb
# api = wandb.Api()
# runs = api.runs(f'{WANDB_ENTITY}/{WANDB_PROJECT}')  # or 'anonymous/...'
#
# histories = []
# for run in runs:
#     hist = run.history(keys=['epoch','train/acc','val/acc','train/loss','val/loss'])
#     histories.append({
#         'run_name'    : run.name,
#         'best_val_acc': run.summary.get('best_val_acc', 0),
#         'config'      : dict(run.config),
#         'history'     : {
#             'epoch'      : hist['epoch'].tolist(),
#             'train_acc'  : hist['train/acc'].tolist(),
#             'val_acc'    : hist['val/acc'].tolist(),
#             'train_loss' : hist['train/loss'].tolist(),
#             'val_loss'   : hist['val/loss'].tolist(),
#         }
#     })
print('W&B API fallback cell — uncomment to use in a fresh session.')


W&B API fallback cell — uncomment to use in a fresh session.


> Run this cell at any time to download **all plots** in one ZIP file.


In [40]:
import glob
pngs = glob.glob(f'{RESULTS_DIR}/*.png')
print(f'Found {len(pngs)} PNG files in {RESULTS_DIR}:')
for p in sorted(pngs): print(f'  {os.path.basename(p)}')

zip_path = '/content/vgg6_all_results.zip'
!zip -j $zip_path $RESULTS_DIR/*.png $RESULTS_DIR/*.csv
colab_files.download(zip_path)
print(f'\n✅  Downloaded: vgg6_all_results.zip  ({len(pngs)} images + CSV)')


Found 14 PNG files in /content/vgg6-cifar10-assignment/results:
  q1_baseline_curves.png
  q2a_activation_bar.png
  q2a_activation_train_loss.png
  q2a_activation_val_acc.png
  q2b_optimizer_bar.png
  q2b_optimizer_train_loss.png
  q2b_optimizer_val_acc.png
  q2c_heatmap.png
  q3_plot1_activations.png
  q3_plot2_optimizers.png
  q3_plot3_act_loss.png
  q3_plot4_opt_loss.png
  q3_scatter_lr_acc.png
  sample_grid.png
  adding: q1_baseline_curves.png (deflated 9%)
  adding: q2a_activation_bar.png (deflated 18%)
  adding: q2a_activation_train_loss.png (deflated 7%)
  adding: q2a_activation_val_acc.png (deflated 6%)
  adding: q2b_optimizer_bar.png (deflated 16%)
  adding: q2b_optimizer_train_loss.png (deflated 8%)
  adding: q2b_optimizer_val_acc.png (deflated 6%)
  adding: q2c_heatmap.png (deflated 13%)
  adding: q3_plot1_activations.png (deflated 6%)
  adding: q3_plot2_optimizers.png (deflated 6%)
  adding: q3_plot3_act_loss.png (deflated 8%)
  adding: q3_plot4_opt_loss.png (deflated 8%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅  Downloaded: vgg6_all_results.zip  (14 images + CSV)


The **Parallel Coordinates** plot is built in the W&B dashboard UI. Follow these steps:

1. Open your W&B project URL (printed after each run above)
2. Click **'+ Add Panel'** → **'Parallel Coordinates'**
3. Add these axes (in order):
   - `config/activation`
   - `config/optimizer`
   - `config/lr`
   - `config/batch_size`
   - `config/num_epochs`
   - `summary/best_val_acc`  ← colour this axis
4. Click **'Done'**
5. Export: click **⋮ → Download → PNG**

> 💡 Filter to only Q2a runs using `Run name contains 'act='` in the W&B filter bar.


In [ ]:
# Print all W&B run URLs for easy navigation
print('📊  All W&B Run URLs:')
print('─' * 60)
for r in q2a_results + q2b_results + q2c_results:
    print(f"  {r['run_name'][:50]:50s}  {r['run_url']}")
